In [2]:
import pandas as pd
import numpy as np

In [3]:
movies = pd.read_csv("../data/movies.csv")
ratings = pd.read_csv("../data/ratings.csv")

print("Movies:", movies.shape)
print("Ratings:", ratings.shape)

movies.head()

Movies: (9742, 3)
Ratings: (100836, 4)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
movies['genres'] = movies['genres'].str.replace('|', ' ')

movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy Romance
3,4,Waiting to Exhale (1995),Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(movies['genres'])

print(tfidf_matrix.shape)

(9742, 23)


In [6]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

cosine_sim.shape

(9742, 9742)

In [7]:
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

indices.head()

title
Toy Story (1995)                      0
Jumanji (1995)                        1
Grumpier Old Men (1995)               2
Waiting to Exhale (1995)              3
Father of the Bride Part II (1995)    4
dtype: int64

In [8]:
def recommend_movies(title, cosine_sim=cosine_sim):

    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:6]

    movie_indices = [i[0] for i in sim_scores]

    return movies['title'].iloc[movie_indices]

In [9]:
recommend_movies("Toy Story (1995)")

1706                                       Antz (1998)
2355                                Toy Story 2 (1999)
2809    Adventures of Rocky and Bullwinkle, The (2000)
3000                  Emperor's New Groove, The (2000)
3568                             Monsters, Inc. (2001)
Name: title, dtype: object

In [10]:
user_movie_matrix = ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

user_movie_matrix.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
user_movie_matrix_filled = user_movie_matrix.fillna(0)

In [12]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(user_movie_matrix_filled)

user_similarity.shape

(610, 610)

In [13]:
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.index
)

user_similarity_df.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
userId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.027283,0.059720,0.194395,0.129080,0.128152,0.158744,0.136968,0.064263,0.016875,...,0.080554,0.164455,0.221486,0.070669,0.153625,0.164191,0.269389,0.291097,0.093572,0.145321
2,0.027283,1.000000,0.000000,0.003726,0.016614,0.025333,0.027585,0.027257,0.000000,0.067445,...,0.202671,0.016866,0.011997,0.000000,0.000000,0.028429,0.012948,0.046211,0.027565,0.102427
3,0.059720,0.000000,1.000000,0.002251,0.005020,0.003936,0.000000,0.004941,0.000000,0.000000,...,0.005048,0.004892,0.024992,0.000000,0.010694,0.012993,0.019247,0.021128,0.000000,0.032119
4,0.194395,0.003726,0.002251,1.000000,0.128659,0.088491,0.115120,0.062969,0.011361,0.031163,...,0.085938,0.128273,0.307973,0.052985,0.084584,0.200395,0.131746,0.149858,0.032198,0.107683
5,0.129080,0.016614,0.005020,0.128659,1.000000,0.300349,0.108342,0.429075,0.000000,0.030611,...,0.068048,0.418747,0.110148,0.258773,0.148758,0.106435,0.152866,0.135535,0.261232,0.060792


In [14]:
def recommend_movies_for_user(user_id, n_recommendations=5):

    similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:11]

    user_movies = user_movie_matrix.loc[user_id]
    watched_movies = user_movies[user_movies.notna()].index

    similar_users_ratings = user_movie_matrix.loc[similar_users.index]

    movie_scores = similar_users_ratings.mean(axis=0)

    movie_scores = movie_scores.drop(watched_movies)

    recommendations = movie_scores.sort_values(ascending=False).head(n_recommendations)

    return recommendations

In [15]:
recommend_movies_for_user(1)

movieId
53322    5.0
4014     5.0
3100     5.0
4015     5.0
3972     5.0
dtype: float64

In [16]:
def recommend_movie_titles(user_id, n_recommendations=5):

    recommendations = recommend_movies_for_user(user_id, n_recommendations)

    movie_ids = recommendations.index

    recommended_titles = movies[movies["movieId"].isin(movie_ids)][["title"]]

    return recommended_titles

In [17]:
recommend_movie_titles(1)

,title
2342,"River Runs Through It, A (1992)"
2963,"Legend of Drunken Master, The (Jui kuen II) (1..."
2998,Chocolat (2000)
2999,"Dude, Where's My Car? (2000)"
6497,Ocean's Thirteen (2007)


In [18]:
def hybrid_recommendation(user_id, movie_title, n=10):

    # Content based recommendations
    content_rec = recommend_movies(movie_title)

    # Collaborative filtering recommendations
    collab_rec = recommend_movie_titles(user_id)

    # Combine both
    hybrid = pd.concat([
        pd.DataFrame(content_rec, columns=["title"]),
        collab_rec
    ])

    # Remove duplicates
    hybrid = hybrid.drop_duplicates()

    return hybrid.head(n)

In [19]:
hybrid_recommendation(1, "Toy Story (1995)")

,title
1706,Antz (1998)
2355,Toy Story 2 (1999)
2809,"Adventures of Rocky and Bullwinkle, The (2000)"
3000,"Emperor's New Groove, The (2000)"
3568,"Monsters, Inc. (2001)"
2342,"River Runs Through It, A (1992)"
2963,"Legend of Drunken Master, The (Jui kuen II) (1..."
2998,Chocolat (2000)
2999,"Dude, Where's My Car? (2000)"
6497,Ocean's Thirteen (2007)
